# Practical 2: Whole-Farm Google GeoAI Agronomic Advisory
### Hands-on Session: Direct, Farmer-Friendly Agronomic Intelligence with Google Gemini

In this practical:
- Connect directly to Google Gemini using the API key in `.env`.
- Ingest farm remote sensing telemetry into the AI context.
- Generate an authoritative, plain-language Farmer Advisory Report (no math or index jargon).
- Query the GeoAI Assistant with real-world farming questions cell-by-cell.

## Step 1: Import required libraries

In [1]:
import os
import sys
import json
import pandas as pd
from IPython.display import display, Markdown
from dotenv import load_dotenv
from google import genai

## Step 2: Initialize Google GenAI Client with your API key from .env

In [2]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
model_name = os.getenv("GEOAI_MODEL", "gemini-3.5-flash-lite")

client = genai.Client(api_key=gemini_api_key)
print("Google GenAI Client successfully initialized.")
print("Target Model:", model_name)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Google GenAI Client successfully initialized.
Target Model: gemini-3.5-flash-lite


## Step 3: Load farm condition context for the AI

In [3]:
df_ts = pd.read_parquet("data/rice_farm_timeseries.parquet")

agg = df_ts.groupby('date')[['NDVI', 'NDRE', 'LSWI']].mean().reset_index()
max_row = agg.loc[agg['NDVI'].idxmax()]
latest_row = agg.iloc[-1]

farm_context = {
    "total_monitored_plots": int(df_ts['plot_id'].nunique()),
    "peak_vegetative_date": max_row['date'].strftime('%Y-%m-%d'),
    "latest_observation_date": latest_row['date'].strftime('%Y-%m-%d'),
    "growth_trend": "Crop has transitioned from peak vegetative green canopy to natural grain ripening and drying down",
    "field_moisture_trend": "Field moisture has dropped significantly as plants reach maturity"
}

print("Farm Telemetry Context for Google GeoAI:")
print(json.dumps(farm_context, indent=2))

Farm Telemetry Context for Google GeoAI:
{
  "total_monitored_plots": 17,
  "peak_vegetative_date": "2024-11-04",
  "latest_observation_date": "2024-12-14",
  "growth_trend": "Crop has transitioned from peak vegetative green canopy to natural grain ripening and drying down",
  "field_moisture_trend": "Field moisture has dropped significantly as plants reach maturity"
}


## Step 4: Generate Live Plain-Language Farmer Advisory Report
We instruct Google Gemini to act as a practical extension officer talking to an everyday Nigerian rice farmer.

In [4]:
prompt = f"""You are an experienced, highly practical Rice Agronomy Extension Officer talking directly to an everyday Nigerian rice farmer.
The farmer does not know or care about decimal numbers like NDVI or cryptic satellite indices. They want clear, plain-language advice they can use on their farm today.

Farm context:
{json.dumps(farm_context, indent=2)}

Write a practical Farmer Advisory Report covering:
1. Current State of Your Farm: Explain what stage the rice is in right now and whether the crop is healthy.
2. Fertilizer Advice: Tell the farmer plainly whether they should buy/apply Urea or any fertilizer now, or if it would be a total waste of money.
3. Water Control: Give clear instructions on what to do with the canal water today (should they pump more water or drain it completely?).
4. Harvest Preparations: Tell the farmer when the rice will be ready for cutting and when to arrange combine harvesters.
5. Immediate Actions for Today: A simple 3-point checklist for farm workers.

Format in clean Markdown with clear headings and bullet points. Do not use emojis."""

print("Calling Google Gemini GeoAI API...")
report = client.models.generate_content(model=model_name, contents=prompt)
display(Markdown(report.text))

Calling Google Gemini GeoAI API...


# Farmer Advisory Report: Rice Crop Status and Pre-Harvest Guide

## 1. Current State of Your Farm
Your rice across all 17 monitored plots is doing very well and is fully on track. 

We reached the peak of the green, leafy stage around November 4th. Since then, the crop has naturally transitioned into the grain ripening and drying-down phase. The plants have successfully done their heavy growing work, and the grains are now hardening and turning golden. The field is healthy, maturing naturally, and nearing the finish line.

## 2. Fertilizer Advice
**Stop all fertilizer applications immediately. Do not buy or apply Urea or any other fertilizer now.**

Applying fertilizer at this late stage would be a complete waste of your money. The rice has stopped growing new leaves and stalks; it is only filling and drying its grains now. Extra fertilizer at this point will not increase your yield, and it can actually harm the crop by causing lodging (falling over) or delaying uneven maturity. Keep your money in your pocket.

## 3. Water Control
**Drain the fields completely.**

Field moisture has dropped significantly, which is exactly what we want at this stage. You should now cut off all water coming into the plots through the canals. If there is any remaining standing water, open your drainage channels to let it run out completely. 

Allowing the soil to dry out hardens the grains, ensures the crop matures evenly, and makes it much easier for you and your workers (or machines) to move through the field during harvest.

## 4. Harvest Preparations
Your rice is very close to maturity. Based on our observations up to December 14th, you should prepare to start harvesting within the next 1 to 2 weeks. 

* **Check the grains:** Walk into the plots and bite a few grains. If they are hard and firm (not milky), the crop is ready.
* **Book your machinery now:** If you use a combine harvester or threshing machines, contact your operators today to lock in your dates. Do not wait until the day of cutting, or you may face delays while your rice becomes over-ripe and shatters in the field.
* **Prepare storage:** Clean out your store rooms, buy your jute bags, and ensure your tarpaulins are ready for drying the harvested paddy.

## 5. Immediate Actions for Today
Instruct your farm workers to carry out these three tasks today:

1. **Clear Drainage Outlets:** Ensure all bunds and channels are open so any remaining water can drain out of the plots completely.
2. **Scare Away Birds:** As the grains dry and sweeten, bird pressure will be at its highest. Keep guards active in the fields from early morning until evening.
3. **Inspect Field Fences and Access:** Check the paths leading to your plots to ensure trucks, tractors, or labour can easily reach the farm when harvest day arrives.

## Step 5: Define interactive Q&A helper for practical farmer questions

In [5]:
def ask_farmer_ai(question):
    p = f"""You are an experienced Rice Farm Agronomist talking to a practical commercial rice farmer.
Do not use technical satellite jargon or decimal index numbers. Give direct, realistic, practical farming advice.
Farm context: {json.dumps(farm_context)}

Farmer asks: "{question}"
Answer directly and practically in 2-3 short paragraphs. Do not use emojis."""
    r = client.models.generate_content(model=model_name, contents=p)
    return r.text

print("ask_farmer_ai function ready.")

ask_farmer_ai function ready.


## Step 6: Farmer Question 1: Water Control
*What should I do with the water in my fields today? Should I pump more water in or open the canals to drain it out?*

In [6]:
q1 = "What should I do with the water in my fields today? Should I pump more water in or open the canals to drain it out?"
print("Farmer asks:", q1)
display(Markdown(ask_farmer_ai(q1)))

Farmer asks: What should I do with the water in my fields today? Should I pump more water in or open the canals to drain it out?


Open your canals and drain the fields completely right now. Your crop reached peak growth about six weeks ago and has naturally transitioned into the grain ripening and drying down phase. 

Keeping water in the fields at this stage is the worst thing you could do. Draining them out will help even out the maturity of the grain, stop any late tillering, and give your ground the firmness it needs to support your harvester when it is time to cut. 

Leave the fields open and let them dry out thoroughly. Dry soil conditions right now will save you from getting heavy machinery stuck and ensure you get clean, quality grain in the bin.

## Step 7: Farmer Question 2: Fertilizer Investment
*Should I buy and put Urea fertilizer on the farm now, or will it be a waste of money?*

In [7]:
q2 = "Should I buy and put Urea fertilizer on the farm now, or will it be a waste of money?"
print("Farmer asks:", q2)
display(Markdown(ask_farmer_ai(q2)))

Farmer asks: Should I buy and put Urea fertilizer on the farm now, or will it be a waste of money?


Save your money. Putting Urea on the farm right now would be a complete waste. 

Your crop hit peak green canopy back in early November and has naturally transitioned into grain ripening and drying down. At this late stage, the plants are done growing leaves and stems, so they cannot use extra nitrogen to build yield. 

Applying Urea now will not help the grain fill out, and it risks causing uneven ripening or lodging just as you are getting ready for harvest. Keep your wallet closed and focus your efforts on preparing for the combine.